# Dataset 
The dataset `TabularDataset` is built on top of huggingface datasets, which is documented here: https://huggingface.co/docs/datasets/en/index

In contrast to a pure HuggingFace dataset however, `TabularDataset` always returns torch tensors from its `__getitem__` method. These are only converted to numpy when we need to afterwards for working with pure sklearn estimators.

In this notebook we will show how to apply preprocessing, filtering, and on-the-fly transforms to the dataset and show it's indexing and compatability methods for sklearn. 


## Construction

For constructing a dataset we minimally need a path to the directory where data live, the data format they are stored in, and the column in the data that represents the label, or a list thereof. 

In [ ]:
%load_ext autoreload
%autoreload 2

from GalaxySpectrumClassifier import TabularDataset

dataset = TabularDataset(
    path="../data/bronze/default",
    data_format="csv",
    label_columns="source",
)
dataset

We can also construct a dataset from a yaml config file with the same content:

In [ ]:
from pprint import pprint

import yaml

with open("../configs/dataset_example.yaml", "r") as f:
    config = yaml.safe_load(f)

pprint("config file: ")
pprint(config)

dataset = TabularDataset.from_config(config)
dataset

## Indexing 

A single line in the dataset corresponds to a single data point. Indexing hence works as one would expect

In [ ]:
# get a single (the 0th) observation

dataset[0]

notice that we get back a tuple of two torch tensors. The first is the feature tensor, containing the columns of a single observation in the same order they are in the dataset's raw data. The second is the label column, in this case it's called 'source', and it encodes 2 classes: 0 (AGN) and 1 (star formation).

In [ ]:
X, y = dataset[0]

we can slice or use wrap-around indexing too:

In [ ]:
X, y = dataset[10:15]
X, y

In [ ]:
X.shape, y.shape

This gives us rows, i.e., datapoints 10,11,12,13,14. Indexing with -1 gives us the last datapoint, and so on. 

In [ ]:
X, y = dataset[-1]
X, y

We can split the complete dataset into feature and label tensors and  convert them to numpy to use them in a sklearn estimator, which wants numpy arrays:  

In [ ]:
X, y = dataset[:]

X = X.numpy()
y = y.numpy()
X.shape, y.shape, type(X), type(y)

We can also inspect the columns and data sizes of the dataset via the underlying backend object: 

In [ ]:
dataset.backend.column_names

In [ ]:
dataset.backend.num_columns

In [ ]:
dataset.backend.num_rows

Because this is a relatively common thing (sklearn does not train in batches) we have a convenience function for it: 

In [ ]:
from GalaxySpectrumClassifier import TabularDataset, to_xy

dataset = TabularDataset(
    path="../data/silver/default",
    data_format="csv",
    label_columns="source",
)

X, y = to_xy(dataset)
X.shape, y.shape, type(X), type(y)

## Formatting

Selecting of columns and formatting works via the `set_format` function of HuggingFace Datasets as documented here: https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.set_format  

In [ ]:
from GalaxySpectrumClassifier import TabularDataset

dataset = TabularDataset(
    path="../data/silver/default",
    data_format="csv",
    label_columns="source",
)

dataset.set_format(
    columns=[
        "12+log(O/H)",
        "OII_3727",
        "source",
    ]
)

X, y = dataset[1]

X, y

The format will include the missing label columns if they are missing, so we always know what the labels are when indexing.

In [ ]:
from GalaxySpectrumClassifier import TabularDataset

dataset = TabularDataset(
    path="../data/silver/default",
    data_format="csv",
    label_columns="source",
)

dataset.set_format(
    columns=[
        "12+log(O/H)",
        "OII_3727",
        # "source", The label column is missing now!
    ]
)

X, y = dataset[1]  # but the dataset got your back :)
X, y

getting rid of it works as well. 

In [ ]:
dataset.reset_format()
X, y = dataset[1]  # but the dataset got your back :)
X, y

## Filtering and preprocessing

`pre_filter` and `pre_transform` are applied once, when the dataset is constructed. A filter keeps rows for which it returns `True`; a pre-transform can add or replace columns in every remaining row. `TabularDataset` delegates these operations to Hugging Face Datasets, including its fingerprint-based cache.

The example below keeps rows whose `OII_3727` emission-line value is at most `2.0`, then doubles that column. Hook names use dotted import paths; functions defined in this notebook are available under `__main__`.


In [ ]:
from GalaxySpectrumClassifier import TabularDataset


def keep_oii_below(row, maximum):
    return row["OII_3727"] <= maximum


def scale_column(row, column, factor):
    return {column: row[column] * factor}


raw_dataset = TabularDataset(
    path="../data/silver/default",
    data_format="csv",
    label_columns="source",
)

prepared_dataset = TabularDataset(
    path="../data/silver/default",
    data_format="csv",
    label_columns="source",
    pre_filter="__main__.keep_oii_below",
    pre_filter_kwargs={"fn_kwargs": {"maximum": 2.0}},
    pre_transform="__main__.scale_column",
    pre_transform_kwargs={"fn_kwargs": {"column": "OII_3727", "factor": 2.0}},
)

first_retained = next(
    row for row in raw_dataset.backend if keep_oii_below(row, maximum=2.0)
)
print(f"Rows: {len(raw_dataset)} -> {len(prepared_dataset)}")
print(
    f"First retained OII_3727 value: {first_retained['OII_3727']:.4f} -> "
    f"{prepared_dataset.backend[0]['OII_3727']:.4f}"
)

The same setup can live in YAML. `dataset_preprocessing_example.yaml` names the hooks, passes their arguments through Hugging Face's `fn_kwargs`, and sets `hf_dataset_kwargs.cache_dir`. That directory holds the loaded dataset and the Arrow files produced by `filter` and `map`. Relative paths are resolved from the process's working directory; because this notebook runs from `notebooks/`, `../.cache/huggingface/datasets` creates a project-local cache.

The hook functions still need to be importable when the config is loaded; here they are the two functions defined above.


In [ ]:
from pathlib import Path
from pprint import pprint

import yaml

with open("../configs/dataset_preprocessing_example.yaml", "r") as f:
    preprocessing_config = yaml.safe_load(f)

pprint(preprocessing_config)
cache_dir = Path(preprocessing_config["hf_dataset_kwargs"]["cache_dir"]).resolve()
print(f"Hugging Face cache: {cache_dir}")

configured_dataset = TabularDataset.from_config(preprocessing_config)
print(f"Configured dataset rows: {len(configured_dataset)}")
print(f"First doubled value: {configured_dataset.backend[0]['OII_3727']:.4f}")

## Transforming data on the fly before indexing

A regular `transform` is installed with Hugging Face's `with_transform`. It behaves like an output formatter: it runs whenever rows are retrieved instead of rewriting the Arrow data, so its results are not materialized in the `map` cache.

`transform_kwargs["columns"]` selects the columns passed to the formatter. `TabularDataset` also includes the label columns automatically so they remain available. Hugging Face calls the formatter with a batch dictionary—even when one row is requested—so each value below is a list. This example again doubles `OII_3727`, but only `OII_3727` and the automatically included `source` label are exposed by the formatted backend row.

In [ ]:
from GalaxySpectrumClassifier import TabularDataset


def double_selected_oii(batch):
    print(batch)
    batch = dict(batch)
    batch["OII_3727"] = [value * 2 for value in batch["OII_3727"]]
    return batch


on_the_fly_dataset = TabularDataset(
    path="../data/silver/default",
    data_format="csv",
    label_columns="source",
    transform=double_selected_oii,
    transform_kwargs={"columns": ["OII_3727"]},
)

stored_value = raw_dataset.backend[0]["OII_3727"]
formatted_row = on_the_fly_dataset.backend[0]

print(f"Formatted columns: {list(formatted_row)}")
print(f"OII_3727 on retrieval: {stored_value:.4f} -> {formatted_row['OII_3727']:.4f}")
print(f"Stored Arrow value is still: {raw_dataset.backend[0]['OII_3727']:.4f}")